> **日本語版について**: このノートブックは、[jpmorganchase/python-training](https://github.com/jpmorganchase/python-training) の英語原文を日本語に翻訳したものです（Apache License 2.0）。コードの動作は原文から変更していません。

# Web API からのデータ取得

「Web 2.0」の素晴らしい点の1つは、開発者がデータの上にツールを構築できるよう、より形式の整った外部公開用の API が用意されていることです。こうした API は、信頼できるプロトコルを介してデータへの信頼できる接続を提供し、機械が読みやすい形式で信頼できる応答を返します。世の中には数多くの API があります。開発者キーへの登録が必要なものもあれば（セントルイス連邦準備銀行の API は[こちら](https://research.stlouisfed.org/docs/api/)をご覧ください）、無料で使えるものもあります（Coindesk の API は[こちら](https://www.coindesk.com/api)をご覧ください）。ProgrammableWeb には優れた[ディレクトリ](https://www.programmableweb.com/apis/directory)があります。

## 大いなる力には、大いなる責任が伴います。プログラムから Web サイトにアクセスするときは、責任ある礼儀正しいインターネット市民でありましょう

以下では、Coindesk の API を使ってビットコインの日次価格を取得します。Coindesk の API は、他の多くの API と同様に JSON（JavaScript Object Notation）オブジェクトを返しますが、これは簡単にデータフレームに変換できます。ここでは `json` ライブラリと `requests` ライブラリが必要です。

In [ ]:
import datetime
import pandas as pd
import json
import requests

coindeskURL = 'https://api.coindesk.com/v1/bpi/historical/close.json?'

# API から取得
start = datetime.date(2019, 1 ,1)
end = datetime.date(2019, 7, 1)

url = f'{coindeskURL}start={start:%Y-%m-%d}&end={end:%Y-%m-%d}'

print(f'Hitting this url:{url}')

result = requests.get(url)
result.content

これをもう少し掘り下げてみると、データにアクセスするうえで重要な要素がいくつかあります。すべての API が同じ構造のデータを返すわけではありません。多くの場合、ブラウザで URL にアクセスするだけで、返される内容を確認できます。やってみましょう。

[https://api.coindesk.com/v1/bpi/historical/close.json?start=2019-01-01&end=2019-07-01](https://api.coindesk.com/v1/bpi/historical/close.json?start=2019-01-01&end=2019-07-01)

この場合は、次のような結果が得られます：
```
{
    "bpi":
    {
        "2019-01-01":3869.47,
        "2019-01-02":3941.2167,
        ...
    },
    "disclaimer": "This data was produced from the CoinDesk Bitcoin Price Index.",
    "time":
    {
        "updated":"Jul 2, 2019 00:03:00 UTC",
        "updatedISO":"2019-07-02T00:03:00+00:00"
    }
}
```

`bpi`、`disclaimer`、`time` はすべて、参照できるデータです。私たちが主に注目するのは `bpi` です。API によってデータの分け方は異なります。他の API での例は、この後で見ていきます。

Python で JSON の結果を確認してみましょう。

In [ ]:
jsondata = json.loads(result.content)
jsondata

そして、それを pandas のデータフレームに変換できます！なお、pandas には `read_json` というヘルパーもありますが、JSON の出力の構造によっては適さない場合もあります。

In [ ]:
# JSON は、disclaimer と time の出力も一緒に読み込まれる点に注意してください
data = pd.read_json(result.content)
data

In [ ]:
data = pd.DataFrame({'Bitcoin Price Index': jsondata['bpi']})
data

In [ ]:
%matplotlib inline
data.plot()